In [2]:
!nbdev_new

# Please enter a value for repo
repo = ^C
Traceback (most recent call last):
  File "/usr/local/bin/nbdev_new", line 8, in <module>
    sys.exit(nbdev_new())
  File "/usr/local/lib/python3.10/dist-packages/fastcore/script.py", line 119, in _f
    return tfunc(**merge(args, args_from_prog(func, xtra)))
  File "/usr/local/lib/python3.10/dist-packages/nbdev/cli.py", line 83, in nbdev_new
    nbdev_create_config.__wrapped__(**kwargs)
  File "/usr/local/lib/python3.10/dist-packages/nbdev/config.py", line 176, in nbdev_create_config
    d = _prompt_user(req, inf)
  File "/usr/local/lib/python3.10/dist-packages/nbdev/config.py", line 124, in _prompt_user
    if inf is None: res[k] = input(f'# Please enter a value for {k}\n'+msg)
KeyboardInterrupt


Setup

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Imports
from ast import literal_eval

import torch
import matplotlib.pyplot as plt
import matrepr
import numpy as np
import torch.nn.functional as f
from matplotlib.pyplot import figure
from matrepr import mprint
from pydmd import BOPDMD, DMD, HAVOK, HODMD, FbDMD, HankelDMD, OptDMD
from pydmd.plotter import plot_eigs, plot_summary
from pydmd.preprocessing import hankel_preprocessing
from safetensors.torch import load_model
from torchinfo import summary

from algo.embedding import dehankel, hankel
from data import (
    GaussianDataset,
    LinearGaussianDataset,
    RectangularDataset,
    SpiralDataset,
    YinYangDataset,
)
from models.perceptron import MLP
from saved_weights import parser

In [4]:
torch.set_printoptions(sci_mode=True)
matrepr.params.title = True
matrepr.params.indices = False
matrepr.params.precision = 4
matrepr.cell_align = "left"
%matplotlib inline

Model & Dataset

In [5]:
HIDDEN = 5

In [6]:
file_path = "../saved_weights/model.safetensors"
metadata = parser.safetensors_metadata_parser(file_path=file_path)

model = MLP(
    config=literal_eval(metadata["config"]), nonlinearity=metadata["nonlinearity"]
)
load_model(model, file_path, device="cuda:0")

model

MLP(
  (features): Sequential(
    (0): Linear(in_features=2, out_features=5, bias=False)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=5, bias=True)
    (3): ReLU()
    (4): Linear(in_features=5, out_features=3, bias=True)
    (5): ReLU()
    (6): Linear(in_features=3, out_features=2, bias=False)
  )
)

In [7]:
PROBLEM = metadata["dataset"]
NUM_SAMPLES = 500
NUM_GAUSS_FEATURES = 0

SAMPLE_SIZE = 2

In [8]:
# Build dataset
if PROBLEM == "yinyang":
    dataset = YinYangDataset(size=NUM_SAMPLES)

elif PROBLEM == "rectangular":
    dataset = RectangularDataset(
        num_samples=NUM_SAMPLES, num_extra_features=NUM_GAUSS_FEATURES
    )
else:
    raise NotImplementedError()

In [9]:
all_indices = np.arange(NUM_SAMPLES)
train_indices = np.random.choice(
    dataset.features.shape[0], size=SAMPLE_SIZE, replace=False
)
test_indices = np.random.choice(
    np.setdiff1d(all_indices, train_indices), size=SAMPLE_SIZE, replace=False
)

In [10]:
print("Train Indices")
mprint(train_indices)

print("Test Indices")
mprint(test_indices)

Train Indices
<length=2, 2 'int64' elements, array>
[379, 473]
Test Indices
<length=2, 2 'int64' elements, array>
[246, 432]


In [11]:
mprint(dataset.features[train_indices, :])

<2×2, 'torch.float32' elements, torch.strided>
┌                ┐
│ 0.534   0.1421 │
│ 0.3932  0.5906 │
└                ┘


Get Activations

In [12]:
LAYERS_OF_INTEREST = [model.features[2], model.features[3]]